[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/royce-psdi-crystallm-training/blob/master/notebooks/load_and_generate_colab.ipynb)

# Load and Generate Crystal Structures using CrystaLLM

In this notebook, we will use pretrained CrystaLLM models to generate crystal structures in Crystallographic Information File (CIF) format.

Hopefully by the end of this session you will be able to:
1. load a pretrained CrystaLLM model
2. generate crystal structures using different levels of input information
3. locate and examine the generated CIF files
4. visualise the generated crystal structures
5. explore the additional information used by other prompt levels

In [ ]:
# Colab setup for CrystaLLM-pi
# Before running: Runtime → Change runtime type → T4 GPU or better.
# Run this cell once and then restart the session under the Runtime tab above.

%pip install -q uv

%cd /content
![ -d /content/CrystaLLM-pi/.git ] || git clone --depth 1 https://github.com/C-Bone-UCL/CrystaLLM-pi.git

%cd /content/CrystaLLM-pi

!uv pip install --system -r requirements.txt
!uv pip install --system \
    "git+https://github.com/lematerial/material-hasher.git" \
    "git+https://github.com/KellerJordan/Muon"
!uv pip install --system -e .

import os, sys
print("cwd:", os.getcwd())
print("python:", sys.version)

In [ ]:
# Reinstall torch natively

%pip uninstall -y torchvision torchaudio

import torch
import importlib.util

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("torchvision installed:", importlib.util.find_spec("torchvision") is not None)

In [ ]:
# Notebook imports and environment cleanup
%cd /content/CrystaLLM-pi

import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

os.environ["PYTHONWARNINGS"] = "ignore::FutureWarning,ignore::UserWarning"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import pandas as pd


## Part 1. Generating crystal structures with a pretrained base model

We will start with the pretrained CrystaLLM base model.

First, we will generate a crystal structure using the minimum amount of input. Before adding any generation condition, it is useful to see what the original base model produces on its own.
We will then add more information to the input and see how this changes the generation process.

### 1. Load a pretrained model

In [ ]:
# load the pretrained CrystaLLM base model
BASE_MODEL = "c-bone/CrystaLLM-pi_mp_20_base"
print(f"Base model: {BASE_MODEL}")

### 2.  Create Prompts + Generate CIFs

Now that we have selected the pretrained base model, lets have a look at what it generates without any additional constraints.

We will use a Level 1 prompt. At `level_1`, the prompt only contains `data_`so the model is free to generate both the composition and the crystal structure. Therefore, the model has to decide what to generate from the patterns it learned during pretraining.

I guess we could think of this as asking someone to “draw a crystal structure” without giving any further instructions. The answers may vary, but together they show the model's natural starting distribution.


In [ ]:
from pathlib import Path

BASE_OUTPUT = Path("data/workshop_generation/pretrained_base_output.parquet")
BASE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

The generation proceeds in batches.

`num_return_sequences` controls how many candidates are sampled in each batch

`max_return_attempts` sets the maximum number of batches

`target_valid_cifs` tells the pipeline to stop once the requested number of structures has passed its built-in validity checks\

Here, the model samples 20 candidates per batch and it stops once 20 valid CIFs have been collected, or after 20 batches if the tearget has been reached. Since we are using a Level 1 prompt so the generation is unconditional. Feel free to experiment with these settings and see how they affect the generated outputs :）

Also, “valid” only means that the CIF passes the pipeline's built-in checks. It does not guarantee thermodynamic stability.

In [ ]:
!{sys.executable} _load_and_generate.py \
    --hf_model_path {BASE_MODEL} \
    --level level_1 \
    --num_return_sequences 20 \
    --max_return_attempts 5 \
    --target_valid_cifs 20 \
    --output_parquet {BASE_OUTPUT}


### 3. Visualise the results

The generated structures and their associated information are stored in a Parquet file so we can load the file as a pandas DataFrame.

Lets look at the first few results.

In [ ]:
import pandas as pd

base_results = pd.read_parquet(BASE_OUTPUT)

print(f"Number of valid CIFs generated: {len(base_results)}")
print(base_results.columns.to_list())

base_results[["Material ID", "Prompt", "Generated CIF"]].head()

#### Take a look at the generated structure

The generated CIFs are currently stored as text in the dataframe. We can read them as `pymatgen` structures and plot a few examples to see what the model has produced.

> **Try it:** Change `random_state` and rerun the cell to have a look at a different set of generated structures.

In [ ]:
import random
import matplotlib.pyplot as plt
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from ase.visualize.plot import plot_atoms

samples = base_results["Generated CIF"].sample(n=3,random_state=5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, cif in zip(axes, samples):
    structure = Structure.from_str(cif, fmt="cif")
    atoms = AseAtomsAdaptor.get_atoms(structure)

    plot_atoms(atoms, ax, radii=0.5)
    ax.set_title(
        f"{structure.composition.reduced_formula}\n"
        f"{structure.density:.2f} g/cm³"
    )
    ax.axis("off")

plt.show()

#### Density Distribution

We can also take a look at the density distribution

In [ ]:
base_densities = []

for cif in base_results["Generated CIF"]:
    structure = Structure.from_str(cif, fmt="cif")
    base_densities.append(structure.density)

plt.hist(base_densities, bins=10, color="#4C78A8", edgecolor="white")
plt.xlabel("Density (g/cm³)")
plt.ylabel("Number of structures")
plt.title("Pretrained base model")
plt.grid(False)
plt.show()

### Try a different prompt

In this example, we used `level_1`, so the model started from a minimal prompt and decided both the composition and crystal structure itself.

CrystaLLM can also start with more information:

- `level_1`: minimal prompt
- `level_2`: composition
- `level_3`: composition and atomic information
- `level_4`: composition, atomic information and space group

Try changing the prompt level and see how giving the model more information affects what it generates.

## Part 2. Compare with the toy model

Now let’s make a visual comparison between the two models.

The first model is the pretrained base model. It generated structures without being given any target density.

The second model is the toy model that was fine tuned earlier.

We generated 20 valid structures from each model. Instead of looking at only a few individual crystal structures, we will calculate the density of every generated structure and compare the two density distributions.

If the density conditioning is working, we would expect the structures generated by the toy model to have densities closer to the target value. Let’s see what happens.